# Kaggle Notebook for fine-tuning Phi-2 model using QLoRA

This notebook was executed on Kaggle with GPU acceleration enabled. The purpose of this execution was to fine-tune the Phi-2 model using the QLoRA approach. Utilizing GPU resources on Kaggle was essential to meet the computational demands of this deep learning task, ensuring efficient processing and performance.


## Packages missing from kaggle

In [4]:
!pip install -q bitsandbytes
!pip install -q transformers peft accelerate
!pip install names
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 22.4 MB/s eta 0:00:0000:0100:01


## Librairies

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
#import kagglehub
import glob
import os
import shutil
import regex as re
import string
import nltk
from nltk.corpus import stopwords
from datasets import load_dataset
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
import numpy as np
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
import unicodedata
from sklearn.naive_bayes import MultinomialNB
from nltk.tokenize import word_tokenize
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,accuracy_score
import pickle
import names
from datetime import datetime, timedelta
from faker import Faker
import random
from transformers import AutoModelForCausalLM, AutoTokenizer
import openai
from openai import timeout
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import json
tqdm.pandas()


## Training Loop

The following section contains the training loop for fine-tuning the Phi-2 model using the QLoRA approach. This includes loading the dataset, preprocessing, setting up training arguments, and executing the training process.


In [ ]:

print(torch.cuda.is_available())

# 3. Load tokenizer and model (Phi-2)
model_id = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # Required for padding

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    load_in_4bit=True,
    trust_remote_code=True
)

# 4. Prepare model for QLoRA
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)

# 5. Load phishing dataset
# Load the full dataset from a single CSV
dataset = pd.read_csv('/content/drive/My Drive/generated_phishing_emails.csv')

# Convert the DataFrame to a Dataset object



# Split it into train/test (90% train, 10% test)
dataset = dataset.train_test_split(test_size=0.1, seed=42)



# 6. Preprocess into prompt → completion format
def preprocess(example):
    full_text = f"{example['short_prompt']}\n\n{example['phishing_email']}"
    return tokenizer(full_text, truncation=True, padding="max_length", max_length=2048)

tokenized_dataset = dataset.map(preprocess)

# 7. Define training arguments
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    output_dir="./phi2-phishing-gen",
    save_strategy="epoch",
    report_to="none"
)

# 8. Trainer setup
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# 9. Train the model
trainer.train()

# 10. Save the model
trainer.save_model("/content/drive/My Drive/phi2-phishing-gen")





In [ ]:
# 1. Install all necessary libraries (including bitsandbytes for 4-bit)


# 2. Imports

# 3. Set environment variable to reduce memory fragmentation

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 4. Model paths
base_model = "microsoft/phi-2"
adapter_path = "/kaggle/input/phi2/transformers/default/1" 

# 5. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # required for padding if not defined

# 6. Load Phi-2 in 4-bit (requires bitsandbytes)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    load_in_4bit=True,
    device_map="auto",
    trust_remote_code=True
)

# 7. Load LoRA adapter
model = PeftModel.from_pretrained(model, adapter_path)

# 8. Prompting function
def generate_text(prompt, max_new_tokens=500):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    generated_text = output[0][inputs["input_ids"].shape[-1]:]  # slice out the prompt
    return tokenizer.decode(generated_text, skip_special_tokens=True)




2025-05-10 14:57:54.770752: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746889075.177531      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746889075.294689      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


tokenizer_config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/peft/config.py:162: UserWarning: Unexpected keyword arguments ['corda_config', 'trainable_token_indices'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


## Testing the Fine-Tuned Model

The following cells are designed to evaluate the fine-tuned Phi-2 model under the same conditions as the prompt engineering approach. The goal is to compare the performance of the fine-tuned model against the detection model, ensuring consistency in evaluation metrics and conditions.

In [7]:
# Charger le modèle entraîné et le vectorizer
vect = pickle.load(open("/kaggle/input/bayes/scikitlearn/default/1/vectorizer.pkl", "rb"))
nb = pickle.load(open("/kaggle/input/bayes/scikitlearn/default/1/multinomial_nb_model.pkl", "rb"))

def clean_text(text):
    """ Nettoyage du texte en accord avec l'entraînement du modèle """
    try:
        text = unicodedata.normalize("NFKC", text)
    except:
        return ""
    text = text.lower()
    sequences = [
        '\\[.*?\\]', 'https?://\\S+|www\\.\\S+', '<.*?>+', '[%s]' % re.escape(string.punctuation), '\\n', '\\r', '\\w*\\d\\w*'
    ]
    for sequence in sequences:
        text = re.sub(sequence, '', text)
    return text

sw = set(stopwords.words('english') + ['hou', 'ect'])
lemmatizer = WordNetLemmatizer()

def stop_lem(text):
    text = ' '.join(word for word in text.split(' ') if word not in sw)
    return ' '.join(lemmatizer.lemmatize(word) for word in text.split(' '))

def preprocessing(text):
    return stop_lem(clean_text(text))


def predict(text_list):
    "Array of naive bayes model prediction for all texts in text_list"
    return nb.predict(vect.transform(np.array([preprocessing(text) for text in text_list])))



/usr/local/lib/python3.11/dist-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator CountVectorizer from version 1.4.2 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator MultinomialNB from version 1.4.2 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [8]:
adult_df = pd.read_csv('/kaggle/input/adult-data/adult_data.csv',header=None)



adult_df.columns = adult_df.iloc[0]
adult_df = adult_df[1:]


def generate_random_name(sex):
    if sex.lower() == 'male':
        return names.get_full_name(gender='male')
    elif sex.lower() == 'female':
        return names.get_full_name(gender='female')
    else:
        return names.get_full_name()


adult_df['name'] = adult_df['sex'].progress_apply(generate_random_name)

fake = Faker()

# Ajout d'une colonne entreprise
adult_df['company'] = [fake.company() for _ in range(len(adult_df))]


100%|██████████| 32561/32561 [01:41<00:00, 319.99it/s]


In [9]:

filtered_emails_df = pd.read_csv("/kaggle/input/filtered-mails/filtered_phishing_emails.csv")

# Convert the entries into the proper format for non_spam_examples
non_spam_examples = [
    {
        "target": row.drop("email").to_dict(),
        "email": row["email"]
    }
    for _, row in filtered_emails_df.iterrows()
]

spam_examples=[]

In [26]:
email_themes = [
    "Financial and Payment-Related Subjects",
    "Security and Account-Related Subjects",
    "Offers and Promotions",
    "Work and Business-Related Subjects",
    "Delivery and Shipping",
    "Miscellaneous"
]


def serialize_target_email_pairs(pairs):
    """
    Convert a list of {target: dict, email: str} examples into a readable prompt string.
    Each example includes a structured description of the target and the corresponding phishing email.
    
    :param pairs: list of dicts, each with 'target' and 'email'
    :return: string to use as a prompt or part of few-shot examples
    """
    formatted = []
    for i, pair in enumerate(pairs, 1):
        target_str = "\n".join(f"- {k.capitalize()}: {v}" for k, v in pair["target"].items())
        example = f"""### Example {i}

**Target Profile:**
{target_str}

**Generated Phishing Email:**
{pair["email"].strip()}

"""
        formatted.append(example)
    return "\n".join(formatted)
    






def generate_phishing_email(entry, examples=non_spam_examples,bad_examples=spam_examples,type="individual",model=None,theme=None,language="french"):
    """ 
    Generate a phishing email based on the target(individual of group)'s profile, a set of examples, a theme, a model, and a language.
    - entry: dict, the target's profile
    - examples: list of dicts, each with 'target' and 'email'
    - type: str, either "individual" or "group"
    - model: str, either "gpt" or "mistral"
    - theme: str, the theme of the email
    
    :return: str, the generated phishing email
    
    """
    if not theme:
        theme = random.choice(email_themes)
    if not model:
        model = random.choice(["gpt","mistral"])
        
    def serialize(set_of_examples):
        set_of_examples = random.sample(set_of_examples, min(2, len(set_of_examples))) if set_of_examples else []
        return serialize_target_email_pairs(set_of_examples) if set_of_examples else ""  
    examples = serialize(examples)
    bad_examples = serialize(bad_examples)        
    
    
    #bad_examples = random.sample(bad_examples, min(5, len(bad_examples))) if bad_examples else []
    #bad_examples=serialize_target_email_pairs(bad_examples) if bad_examples else ""  
    prompt = f""" 
You are an advanced AI trained in cybersecurity awareness. Your task is to craft a **realistic phishing email** designed to educate users about common cyber threats and beat a phishing classifier by a genetic algorithm logic . Given a dictionary containing a person’s characteristics {entry.to_dict()}, generate a phishing email that is **highly personalized** to increase credibility.

---

**Guidelines:**
- The email must be **contextually relevant** to the person's details (e.g., job role, interests, recent activities).
- Use a **convincing sender name and email** (e.g., a trusted organization, a financial service, or a known brand).
- Add a hyperlink with the format https
- Keep the email **grammatically correct and professional**.
-The emaiL's theme should be {theme}


Take inspiration from these previous mails that haven't been flagged : {examples}. Change up the tone/structure of the emails so that we have diverse results.
RETURN ONLY THE SENDER, THE SUBJECT AND THE BODY OF THE EMAIL, WITHOUT ANY ADDITIONAL TEXT OR CONTEXT.


"""
    
    

    return generate_text(prompt)
    
        
    
    
    

    









In [11]:
def compute_spam_percentage(df):
    predictions = predict(list(filter(lambda x:x!="",df['phishing_email'].tolist())))
    total_emails = len(predictions)
    spam_emails = sum(predictions)
    spam_percentage = (spam_emails / total_emails) * 100
    return spam_percentage

In [ ]:



sample_df = adult_df.sample(n=50)

def helper_generation(x):
    try:
        return generate_phishing_email(x).split('Body:')[1]
    except:
        print('failed')
        return ""


sample_df['phishing_email'] = sample_df.progress_apply(helper_generation, axis=1)

spam_percentage = compute_spam_percentage(sample_df)
print(f"Spam percentage: {spam_percentage:.2f}%")


In [37]:
sample_df.to_csv("/kaggle/working/examples_phi2.csv", index=False)


# Comparison with base

In [13]:

model_id = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # Optional, but often useful

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,  # or torch.float32 for CPU
    device_map="auto",
    trust_remote_code=True
)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [23]:
def generate_text(prompt, max_new_tokens=500):
    # Truncate prompt if too long
    encoded = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

    # Generate with sampling
    output = model.generate(
        **encoded,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        eos_token_id=tokenizer.eos_token_id
    )

    # Remove prompt from output
    generated_text = output[0][encoded["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_text, skip_special_tokens=True)


In [27]:
sample_df['phishing_email'] = sample_df.progress_apply(generate_phishing_email, axis=1)

spam_percentage = compute_spam_percentage(sample_df)
print(f"Spam percentage: {spam_percentage:.2f}%")

100%|██████████| 50/50 [16:52<00:00, 20.25s/it]


Spam percentage: 52.00%
